# Aula 12 — Exercício prático: usuários de um app de música (sem entrega)

**Análise e Ciência de Dados · Especialização em Ciência de Dados · IFRN**

Um aplicativo de música **fictício** quer entender como seus 240 usuários usam o serviço. Você vai aplicar o roteiro do laboratório: **univariados → bivariados → customização → Plotly**, sempre colocando no gráfico a estatística que sustenta a leitura.

> Dados **sintéticos** (`gerar_dados.py`, semente fixa). Nenhum registro é real. O exercício é para treino e **não precisa ser entregue**. Cada passo traz o **resultado esperado** para você conferir. O gabarito comentado está em `Aula12_Exercicio_Gabarito.ipynb` — tente antes de olhar!

| Coluna | Descrição |
|---|---|
| `usuario_id` | identificador |
| `idade` | idade em anos (16–65) |
| `plano` | Gratuito ou Premium |
| `genero_favorito` | Pop, Rock, Sertanejo, Forró, MPB, Eletrônica |
| `dispositivo` | Celular, Computador, Smart TV |
| `horas_semana` | horas de música ouvidas por semana |
| `playlists_criadas` | número de playlists criadas |
| `anuncios_semana` | anúncios ouvidos por semana (Premium = 0) |
| `satisfacao` | nota de satisfação (0 a 10) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
from scipy import stats
from pathlib import Path

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["axes.titleweight"] = "bold"
pio.renderers.default = "plotly_mimetype+notebook_connected"
Path("figuras").mkdir(exist_ok=True); Path("graficos").mkdir(exist_ok=True)
CORES_PLANO = {"Gratuito": "#9ecae1", "Premium": "#0173b2"}

## Passo 1 — Carregar e conhecer os dados

Leia `dados/app_musica_usuarios.csv`, veja `shape`, `head()`, `describe()` e conte os usuários por plano.

> ✅ **Resultado esperado:** 240 linhas × 9 colunas; **151 Gratuito** e **89 Premium**; idade média ≈ 40,3 anos.

In [ ]:
# dica: pd.read_csv(...), df.shape, df['plano'].value_counts()

# seu código aqui


## Passo 2 — Histograma de `horas_semana` com média e mediana

Calcule média, mediana, desvio-padrão e assimetria. Desenhe o histograma e marque média e mediana com `axvline`, com os valores na legenda.

> ✅ **Resultado esperado:** média ≈ **16,5 h**, mediana ≈ **16,2 h**, desvio-padrão ≈ 4,6 h, assimetria ≈ **0,29** → distribuição quase simétrica (média e mediana quase coincidem — compare com o tempo de entrega da aula, skew = 1,18).

In [ ]:
# dica: sns.histplot(..., ax=ax); ax.axvline(media, label=f'média = {media:.1f} h')

# seu código aqui


## Passo 3 — Boxplot de `horas_semana` por plano, com *n* e medianas

Calcule Q1, mediana, Q3 e IQR de cada plano. Faça o boxplot com `showmeans=True`, escreva a mediana ao lado de cada caixa e o *n* no rótulo do eixo x.

> ✅ **Resultado esperado:** Gratuito: mediana **15,3 h** (Q1 = 12,0; Q3 = 18,65; IQR ≈ 6,7). Premium: mediana **18,5 h** (Q1 = 14,7; Q3 = 21,7; IQR = 7,0). Usuários Premium ouvem ~3 h a mais. No conjunto todo há **1 outlier** pelo critério de Tukey (acima de ≈ 29,8 h).

In [ ]:
# dica: m.groupby('plano')['horas_semana'].quantile([.25, .5, .75]).unstack()

# seu código aqui


## Passo 4 — Violino da `satisfacao` por plano

Use `sns.violinplot(..., inner="quart", cut=0)` e escreva a mediana de cada grupo no gráfico.

> ✅ **Resultado esperado:** Mediana da satisfação: **Gratuito 3,6** e **Premium 7,3**. As duas distribuições quase não se sobrepõem.

In [ ]:
# dica: m.groupby('plano')['satisfacao'].median()

# seu código aqui


## Passo 5 — Dispersão `idade` × `horas_semana` com r e reta de regressão

Calcule Pearson, Spearman e a reta (`stats.linregress`). Desenhe com `scatterplot` (cor = plano) + `regplot(scatter=False)` e escreva r e a equação numa caixa de texto.

> ✅ **Resultado esperado:** Pearson **r ≈ −0,65**, Spearman ρ ≈ −0,66; reta **horas ≈ 25,0 − 0,21 × idade** → a cada 10 anos a mais, cerca de 2,1 h a menos por semana.

In [ ]:
# dica: stats.pearsonr(x, y)[0]; reg = stats.linregress(x, y); reg.slope, reg.intercept

# seu código aqui


## Passo 6 — Barras: horas médias por gênero favorito (IC 95% e *n*)

Ordene os gêneros pela média, use `sns.barplot(..., errorbar=("ci", 95))` e escreva o *n* de cada barra. Destaque o maior valor com outra cor.

> ✅ **Resultado esperado:** **Eletrônica 19,6 h** (n = 30) é o maior; **Rock 15,0 h** (n = 34) o menor; MPB, Forró, Pop e Sertanejo ficam entre 15,9 e 16,7 h, com intervalos de confiança que se sobrepõem bastante.

In [ ]:
# dica: ordem = m.groupby('genero_favorito')['horas_semana'].mean().sort_values(ascending=False).index

# seu código aqui


## Passo 7 — Mapa de calor da correlação

Monte a matriz com as colunas numéricas (sem `usuario_id`), use paleta divergente `vlag` com `center=0`, `vmin=-1`, `vmax=1`, `annot=True` e máscara do triângulo superior. **Desafio:** recalcule anúncios × satisfação **só para o plano Gratuito**.

> ✅ **Resultado esperado:** Par mais forte: **anúncios × satisfação r = −0,82**; depois idade × horas (−0,65) e playlists × horas (0,54). Só no plano Gratuito, anúncios × satisfação cai para **r ≈ −0,42** — parte da correlação geral vem do **plano** (Premium tem 0 anúncios *e* satisfação alta): variável de confusão.

In [ ]:
# dica: corr = m[colunas].corr(); mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# seu código aqui


## Passo 8 — Customização: painel 2×2 para a diretoria

Monte `fig, axes = plt.subplots(2, 2, figsize=(12, 8))` com os gráficos dos passos 2, 3, 5 e 7. Use `fig.suptitle` com a principal conclusão, rodapé com a fonte e salve em `figuras/exercicio_painel.png` com `dpi=200, bbox_inches="tight"`.

> ✅ **Resultado esperado:** Arquivo `figuras/exercicio_painel.png` criado; títulos com conclusões, eixos com unidades, mesma cor para o mesmo plano nos quatro painéis.

In [ ]:
# dica: reaproveite o código dos passos anteriores trocando ax por axes[0, 0], axes[0, 1]...

# seu código aqui


## Passo 9 — Plotly: dispersão interativa e exportação para HTML

Com `px.scatter`, faça idade × horas com `color="plano"` e `hover_data=["genero_favorito", "dispositivo", "satisfacao"]`. Salve em `graficos/exercicio_dispersao.html` (`include_plotlyjs="cdn"`). Faça também um `px.box` da satisfação por plano com `points="all"`.

> ✅ **Resultado esperado:** Ao passar o mouse sobre um ponto aparecem idade, horas, gênero, dispositivo e satisfação; o arquivo HTML abre em qualquer navegador. No boxplot, o *hover* mostra mediana 3,6 (Gratuito) e 7,3 (Premium).

In [ ]:
# dica: fig = px.scatter(...); fig.write_html('graficos/exercicio_dispersao.html', include_plotlyjs='cdn'); fig.show()

# seu código aqui


## Para refletir
1. Em qual passo a **mediana** comunicou melhor que a média? Por quê?
2. O r = −0,82 entre anúncios e satisfação significa que *cada* anúncio reduz a satisfação? O que o desafio do passo 7 mostrou?
3. Qual dos gráficos você colocaria **sozinho** num slide para a diretoria? Que título daria a ele?